# Token-bounded AI analysis
Start with zero-token deterministic processing. Each selected AI task makes at most one call; cache hits use no additional model tokens. The default notebook only plans calls.

In [ ]:
from pathlib import Path
import tempfile
import json
root = Path.cwd()
if not (root / 'examples').exists():
    root = root.parent
assert (root / 'examples/parser_samples.json').exists(), 'Run from the repository or notebooks directory'
from timeline_demo.pipeline import Input, run_pipeline, read_timeline
from timeline_demo.core.manifest import verify_bundle
work = tempfile.TemporaryDirectory()
workdir = Path(work.name)
bundle = workdir / 'bundle'
manifest = run_pipeline([
    Input('cloudtrail', root / 'examples/raw/aws/cloudtrail_real_sample.json'),
    Input('entra_signin', root / 'examples/raw/entra/entra_signin_real_sample.jsonl'),
    Input('crowdstrike_detection', root / 'examples/raw/edr/crowdstrike_detection_real_sample.json'),
], bundle, 'notebook-demo')


In [ ]:
from timeline_demo.ai import Harness, load_policy, prepare
policy = load_policy()
[{ 'task':task, **entry } for task, entry in policy['tasks'].items()]

In [ ]:
plans = {task:prepare(bundle, task, policy) for task in policy['tasks']}
[{ 'task':task, 'model':p['model'], 'input_token_bound':p['input_token_bound'], 'coverage':p['coverage']} for task,p in plans.items()]

The input bound includes the schema and instructions. It is a conservative UTF-8 byte bound plus framing reserve, not a tokenizer estimate. The model sees grouped event examples with short citation IDs; identities are pseudonymized. Review minimization requirements before enabling egress.

In [ ]:
ALLOW_AI = False
if ALLOW_AI:
    # Keep this ledger in a durable, access-controlled location for real cases.
    harness = Harness(workdir/'analysis.sqlite', policy=policy)
    result = harness.run(bundle, task='summarize', allow_ai=True)
    print(result)
    cached = harness.run(bundle, task='summarize', allow_ai=True)
    assert cached['cache_hit']
    assert cached['tokens_used_this_call'] == 0
else:
    print('No model request made. Source evidence stays in the local bundle.')

Run `correlate` or `review` explicitly when the investigation needs them. The harness does not auto-escalate to more expensive models. Do not treat model confidence or syntactically valid citations as proof of an interpretation.

In [ ]:
assert verify_bundle(bundle)['bundle_id'] == manifest['bundle_id']
work.cleanup()